# Notebook setup

Run this cell first. It locates the shared code and sets the working directory
to `Code/`. This notebook retains an earlier experiment implementation.
Use the maintained scripts for new comparable PPO results.

Historical analyses/evaluations require separately supplied studies, metrics or
checkpoints. See the repository's `notebooks/README.md` for required inputs.


In [ ]:
from pathlib import Path
import os
import sys

# Legacy cells use paths relative to Code/.
start = Path.cwd().resolve()
REPO_ROOT = next(
    (candidate for candidate in (start, *start.parents)
     if (candidate / "Code" / "training_settings.ini").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from the repository or a notebook subdirectory.")
CODE_DIR = REPO_ROOT / "Code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
os.chdir(CODE_DIR)
print("Working directory:", Path.cwd())

# Set STUDY_DB to the path of a separately supplied historical study.
STUDY_DB = next((p for p in (CODE_DIR / "ppo_search_120.db", REPO_ROOT / ".local/studies/ppo_search_120.db") if p.is_file()), CODE_DIR / "ppo_search_120.db")


# Expanded PPO Search Analysis

This notebook expands the existing search-analysis workflow by combining **Optuna SQLite analysis** and **Weights & Biases run analysis**.

The goal is not only to identify the best trial, but to understand:

- which hyperparameters matter most
- which architectural choices appear repeatedly among strong trials
- whether strong performance comes from broad robustness or only from a subset of demand levels
- what trade-offs exist between reward, queue length, waiting time, and speed
- which regions of the search space look promising for future runs


In [ ]:
import math
import re
import warnings
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 140

if not STUDY_DB.is_file():
    raise FileNotFoundError(f'Supply the historical study and set STUDY_DB: {STUDY_DB}')
DB_PATH = f'sqlite:///{STUDY_DB.as_posix()}'
STUDY_NAME = 'ppo_sumo_bo_120'
TOPK = 10
TOP_FRACTION = 0.15

study = optuna.load_study(study_name=STUDY_NAME, storage=DB_PATH)
df = study.trials_dataframe(attrs=('number','value','state','params','user_attrs','system_attrs','datetime_start','datetime_complete'))
df['duration_h'] = (pd.to_datetime(df['datetime_complete']) - pd.to_datetime(df['datetime_start'])).dt.total_seconds() / 3600
dfc = df.copy()
completed = dfc[dfc['state'] == 'COMPLETE'].copy()
param_cols = [c for c in dfc.columns if c.startswith('params_')]
num_param_cols = [c for c in param_cols if pd.api.types.is_numeric_dtype(dfc[c])]
cat_param_cols = [c for c in param_cols if c not in num_param_cols]

print('Trials:', len(dfc))
print('Completed:', len(completed))
print('Best value:', study.best_value)
print('Best trial:', study.best_trial.number)


## 1. Search-Level Summary

These views answer whether the search ran cleanly, how expensive trials were, and how quickly the study improved.

In [ ]:
summary = pd.DataFrame({
    'n_trials': [len(dfc)],
    'n_complete': [(dfc['state'] == 'COMPLETE').sum()],
    'n_pruned': [(dfc['state'] == 'PRUNED').sum()],
    'n_fail': [(dfc['state'] == 'FAIL').sum()],
    'best_value': [study.best_value],
    'median_complete_value': [completed['value'].median()],
    'mean_duration_h_complete': [completed['duration_h'].mean()],
})
summary.T


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

dfc['state'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Trial state counts')
axes[0].set_xlabel('State')
axes[0].set_ylabel('Count')

axes[1].plot(completed['number'], completed['value'], marker='o', linewidth=1)
axes[1].set_title('Completed trial objective by trial number')
axes[1].set_xlabel('Trial number')
axes[1].set_ylabel('Objective value')

axes[2].scatter(completed['duration_h'], completed['value'], alpha=0.75)
axes[2].set_title('Trial duration vs objective')
axes[2].set_xlabel('Duration (hours)')
axes[2].set_ylabel('Objective value')

plt.tight_layout()
plt.show()


## 2. Hyperparameter Importance and Search Geometry

These views are useful for deciding where future search effort should go.

### Why These Parameters Matter

- `num_conv_layers`, `num_filter_*`, `kernel_size_*`, and `stride_size_*` control how aggressively the state grid is compressed and which spatial traffic patterns can be represented. In this problem, they determine whether the policy can separate lane-level occupancy structure from broader congestion patterns.
- `lstm_units` controls temporal memory. This matters because the signal controller does not only react to a static snapshot; it benefits from encoding queue growth, dissipation, and short-term demand changes over time.
- `num_mlp_layers` and `mlp_neurons_*` control the capacity of the decision head after the convolutional and recurrent encoders. Larger heads can model more complex policy and value mappings, but may also overfit or become harder to optimize.
- `learning_rate`, `optimizer`, and `L2-reg` affect optimization stability. For PPO in particular, unstable optimizer settings can dominate architectural effects if they lead to noisy or prematurely collapsed updates.
- `K_epochs` and `clip_range` control how strongly PPO reuses each rollout. Larger reuse can improve sample efficiency, but can also move the policy too far from the data-collection policy if clipping is not sufficiently restrictive.
- `gamma` and `lambd` control credit assignment over time. In traffic signal control, these parameters shape whether the agent focuses on immediate queue relief or on longer-term network flow quality.
- `entropy_coef` controls exploration pressure. Low entropy can freeze the controller into brittle timing patterns, while high entropy can prevent convergence to decisive phase choices.

When reading the plots below, keep in mind that some parameters are conditional. For example, `num_filter_1` only matters when the sampled architecture actually has a second convolutional layer. Empty or weak-looking effects can therefore be caused by search sparsity rather than by genuine irrelevance.

In [ ]:
from optuna.importance import get_param_importances

imps = get_param_importances(study)
imp_df = pd.DataFrame({'param': list(imps.keys()), 'importance': list(imps.values())}).sort_values('importance', ascending=True)
plt.figure(figsize=(10, max(5, 0.35 * len(imp_df))))
plt.barh(imp_df['param'], imp_df['importance'], color='darkslateblue')
plt.title('Optuna parameter importances')
plt.xlabel('Importance')
plt.ylabel('Parameter')
plt.tight_layout()
plt.show()
imp_df.sort_values('importance', ascending=False)


In [ ]:
corr_cols = ['value'] + num_param_cols
corr_df = completed[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(corr_df, cmap='coolwarm', center=0)
plt.title('Correlation matrix: objective and numeric hyperparameters')
plt.tight_layout()
plt.show()


In [ ]:
top_cut = max(1, int(math.ceil(len(completed) * TOP_FRACTION)))
top_df = completed.sort_values('value', ascending=False).head(top_cut).copy()

display(top_df[['number', 'value'] + param_cols].head(TOPK))

for col in num_param_cols[:6]:
    plt.figure(figsize=(8, 4))
    sns.kdeplot(completed[col].dropna(), label='all completed', linewidth=2)
    sns.kdeplot(top_df[col].dropna(), label='top fraction', linewidth=2)
    plt.title(f'Distribution shift for {col}')
    plt.legend()
    plt.tight_layout()
    plt.show()


## 3. Architecture Patterns Among Strong Trials

This section highlights recurring design choices in the best part of the search space.

In [ ]:
arch_cols = [c for c in param_cols if any(k in c for k in ['num_conv_layers', 'num_filter_', 'kernel_size_', 'stride_size_', 'lstm_units', 'num_mlp_layers', 'mlp_neurons_'])]
top_arch = top_df[arch_cols].copy()
top_arch = top_arch.dropna(axis=1, how='all')

if top_arch.empty:
    print('No architecture columns with observed values in the selected top-fraction trials.')
else:
    for col in top_arch.columns:
        vc = top_arch[col].dropna().value_counts().sort_index()
        if vc.empty:
            print(f'Skipping {col}: no observed values in the selected top-fraction trials.')
            continue
        plt.figure(figsize=(8, 4))
        vc.plot(kind='bar', color='teal')
        plt.title(f'Top-fraction frequency: {col}')
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.tight_layout()
        plt.show()


### Top-3 Trial Comparison

The next block puts the best three completed trials side by side, including both architecture and PPO optimization parameters. This is useful for checking whether strong trials repeatedly favor the same depth, widths, memory size, or PPO update regime rather than only looking good in isolation.

In [ ]:
top3 = completed.sort_values('value', ascending=False).head(3).copy()

architecture_cols = [
    'params_num_conv_layers', 'params_num_filter_0', 'params_num_filter_1',
    'params_kernel_size_0', 'params_kernel_size_1', 'params_stride_size_0', 'params_stride_size_1',
    'params_lstm_units', 'params_num_mlp_layers', 'params_mlp_neurons_0', 'params_mlp_neurons_1'
]
ppo_cols = [
    'params_learning_rate', 'params_K_epochs', 'params_L2-reg', 'params_gamma',
    'params_lambd', 'params_clip_range', 'params_entropy_coef', 'params_optimizer'
]

selected_cols = ['number', 'value'] + [c for c in architecture_cols + ppo_cols if c in top3.columns]
rename_map = {c: c.replace('params_', '') for c in selected_cols if c.startswith('params_')}

#display(top3[selected_cols].rename(columns=rename_map).reset_index(drop=True))

print(top3[selected_cols].rename(columns=rename_map).reset_index(drop=True))

top3[selected_cols].rename(columns=rename_map).reset_index(drop=True).to_csv("./top3_arch.csv", index=False)

shared_rows = []
for col in [c for c in selected_cols if c.startswith('params_')]:
    vals = top3[col].dropna()
    if vals.empty:
        continue
    counts = vals.value_counts()
    if counts.iloc[0] >= 2:
        shared_rows.append({
            'parameter': col.replace('params_', ''),
            'most_common_value': counts.index[0],
            'count_within_top3': int(counts.iloc[0]),
        })

shared_df = pd.DataFrame(shared_rows).sort_values(['count_within_top3', 'parameter'], ascending=[False, True]) if shared_rows else pd.DataFrame(columns=['parameter', 'most_common_value', 'count_within_top3'])
display(shared_df)


## 4. W&B Run Analysis

This section enriches the SQLite search analysis with the per-demand evaluation metrics logged to Weights & Biases.

Edit the `ENTITY` and `PROJECT` values if needed.

In [ ]:
import wandb

ENTITY = 'jzapanag'
PROJECT = 'TrafficSignalControl'
OBJECTIVE_KEY = 'eval/weighted_avg'
MODE = 'max'
TOPK_RUNS = 10
WB_FILTERS = {'state': 'finished'}

api = wandb.Api(timeout=60)
runs = api.runs(f'{ENTITY}/{PROJECT}', filters=WB_FILTERS)

def family(summary, prefix):
    out = {}
    pat = re.compile(rf'^{re.escape(prefix)}(\d+)$')
    for k, v in (summary or {}).items():
        if not isinstance(k, str):
            continue
        m = pat.match(k)
        if m:
            try:
                out[int(m.group(1))] = float(v)
            except Exception:
                pass
    return out

rows = []
for r in runs:
    summ = dict(r.summary or {})
    cfg = dict(r.config or {})
    search_space = cfg.get('search_space', {})
    obj = summ.get(OBJECTIVE_KEY)
    if obj is None:
        continue
    rows.append({
        'name': r.name,
        'run_id': r.id,
        'url': f'https://wandb.ai/{ENTITY}/{PROJECT}/runs/{r.id}',
        OBJECTIVE_KEY: float(obj),
        'score_by_vol': family(summ, 'eval/score@'),
        'speed_by_vol': family(summ, 'eval/avg_speed@'),
        'queue_by_vol': family(summ, 'eval/avg_queue_length@'),
        'wait_by_vol': family(summ, 'eval/cumulative_wait@'),
        **{f'cfg_{k}': v for k, v in search_space.items() if isinstance(v, (int, float, str, bool)) or v is None}
    })

wandb_df = pd.DataFrame(rows)
ascending = MODE.lower() == 'min'
top_runs = wandb_df.sort_values(OBJECTIVE_KEY, ascending=ascending).head(TOPK_RUNS).reset_index(drop=True)
top_runs[['name', 'run_id', OBJECTIVE_KEY, 'url']]


In [ ]:
def expand_metric(df, dict_col, prefix):
    all_vols = sorted(set(v for d in df[dict_col] for v in (d or {}).keys()))
    out = df.copy()
    for vol in all_vols:
        out[f'{prefix}@{vol}'] = out[dict_col].apply(lambda d: d.get(vol, np.nan) if isinstance(d, dict) else np.nan)
    return out, all_vols

top_runs_exp, vols = expand_metric(top_runs, 'score_by_vol', 'score')
top_runs_exp, _ = expand_metric(top_runs_exp, 'speed_by_vol', 'speed')
top_runs_exp, _ = expand_metric(top_runs_exp, 'queue_by_vol', 'queue')
top_runs_exp, _ = expand_metric(top_runs_exp, 'wait_by_vol', 'wait')

def plot_family_lines(df, prefix, ylabel, title):
    cols = [c for c in df.columns if c.startswith(prefix + '@')]
    vols = sorted(int(c.split('@')[1]) for c in cols)
    cols = [f'{prefix}@{v}' for v in vols]
    plt.figure(figsize=(12, 6))
    for _, row in df.iterrows():
        plt.plot(vols, row[cols].astype(float).values, marker='o', label=row['name'])
    plt.title(title)
    plt.xlabel('Traffic volume')
    plt.ylabel(ylabel)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

plot_family_lines(top_runs_exp, 'score', 'Average reward / score', 'Top W&B runs: score vs demand')
plot_family_lines(top_runs_exp, 'speed', 'Average speed', 'Top W&B runs: speed vs demand')
plot_family_lines(top_runs_exp, 'queue', 'Average queue length', 'Top W&B runs: queue vs demand')
plot_family_lines(top_runs_exp, 'wait', 'Cumulative wait', 'Top W&B runs: cumulative wait vs demand')


In [ ]:
def robustness_features(row):
    vols = sorted(row['score_by_vol'].keys())
    scores = np.array([row['score_by_vol'][v] for v in vols], dtype=float)
    x = np.array(vols, dtype=float)
    slope = np.polyfit(x, scores, 1)[0] if len(scores) >= 2 else np.nan
    return pd.Series({
        'score_mean': scores.mean(),
        'score_std': scores.std(),
        'score_min': scores.min(),
        'score_max': scores.max(),
        'score_range': scores.max() - scores.min(),
        'score_slope': slope,
    })

robust_df = pd.concat([top_runs[['name', OBJECTIVE_KEY]], top_runs.apply(robustness_features, axis=1)], axis=1)
display(robust_df.sort_values(OBJECTIVE_KEY, ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].scatter(robust_df['score_mean'], robust_df['score_min'])
axes[0].set_title('Mean score vs worst-demand score')
axes[0].set_xlabel('Mean score')
axes[0].set_ylabel('Worst-demand score')

axes[1].scatter(robust_df['score_std'], robust_df[OBJECTIVE_KEY])
axes[1].set_title('Score variability vs weighted objective')
axes[1].set_xlabel('Score std across demand')
axes[1].set_ylabel('Weighted objective')

axes[2].scatter(robust_df['score_slope'], robust_df[OBJECTIVE_KEY])
axes[2].set_title('Demand sensitivity vs weighted objective')
axes[2].set_xlabel('Linear slope of score over demand')
axes[2].set_ylabel('Weighted objective')

plt.tight_layout()
plt.show()


## 5. Opportunities for Additional Analysis

The most promising additions, depending on the final paper direction, are:

- compare **top-fraction parameter distributions** against the full search space to identify stable design preferences
- compare **best weighted score** against **worst-demand score** to separate peak performance from robustness
- analyze **trial duration vs score** to quantify cost-quality trade-offs
- inspect whether some architectural choices systematically reduce queue length or waiting time even when reward is similar
- perform **top-trial family plots** for score, speed, queue, and wait to identify cross-demand behavior patterns
- compute robustness descriptors such as score standard deviation, slope over demand, and minimum-demand performance
- analyze **pruned vs completed** regions of the search space to avoid repeatedly sampling weak settings in future studies

These views are especially useful for identifying next-step improvements rather than only reporting the best run.